# Meridian M&A Intelligence Platform
## Interactive Live Dashboard

---

### Real-Time Screening with Interactive Controls

This notebook provides an interactive dashboard for real-time M&A target screening and analysis using Jupyter widgets.

In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

# Import Meridian modules
import sys
sys.path.append('..')
from meridian import SyntheticDataGenerator, ScreeningEngine, ValuationEngine, IntelligenceAnalyzer
from meridian.visualizations import *

# Initialize components
print("Meridian Interactive Dashboard v1.0")
print("="*50)
print("Loading components...")

# Load data
import os
if not os.path.exists('../data/synthetic_universe.parquet'):
    generator = SyntheticDataGenerator(seed=42)
    companies = generator.generate_company_universe(5000)
    companies.to_parquet('../data/synthetic_universe.parquet')
else:
    companies = pd.read_parquet('../data/synthetic_universe.parquet')

# Initialize engines
screener = ScreeningEngine(companies)
valuation_engine = ValuationEngine()
analyzer = IntelligenceAnalyzer()

print(f"✓ {len(companies):,} companies loaded")
print("✓ All systems ready")
print("\nUse the controls below to screen targets interactively.")

## 1. Interactive Screening Controls

Adjust parameters in real-time to find optimal targets.

In [ ]:
# Create interactive controls
style = {'description_width': 'initial'}

# Revenue controls
revenue_slider = widgets.IntRangeSlider(
    value=[20, 200],
    min=5,
    max=1000,
    step=10,
    description='Revenue Range ($M):',
    style=style,
    layout=widgets.Layout(width='500px')
)

# Growth rate control
growth_slider = widgets.FloatSlider(
    value=0.25,
    min=0,
    max=1.0,
    step=0.05,
    description='Min Growth Rate:',
    style=style,
    layout=widgets.Layout(width='500px')
)

# Industry selection
industry_select = widgets.SelectMultiple(
    options=companies['industry'].unique(),
    value=['SaaS', 'Fintech'],
    description='Industries:',
    style=style,
    layout=widgets.Layout(width='300px', height='100px')
)

# EBITDA margin control
ebitda_slider = widgets.FloatSlider(
    value=-0.05,
    min=-0.20,
    max=0.30,
    step=0.05,
    description='Min EBITDA Margin:',
    style=style,
    layout=widgets.Layout(width='500px')
)

# Profitability requirement
profitable_checkbox = widgets.Checkbox(
    value=False,
    description='Profitable Only',
    style=style
)

# Exclude distressed
exclude_distressed = widgets.Checkbox(
    value=True,
    description='Exclude Distressed',
    style=style
)

# Run button
run_button = widgets.Button(
    description='Run Screening',
    button_style='primary',
    icon='search'
)

# Output area
output = widgets.Output()

# Results storage
current_results = None

def on_run_clicked(b):
    global current_results
    
    with output:
        clear_output(wait=True)
        
        # Get parameters
        min_rev, max_rev = revenue_slider.value
        
        print("Running screening...")
        
        # Run screening
        results = screener.screen(
            min_revenue=min_rev * 1_000_000,
            max_revenue=max_rev * 1_000_000,
            min_growth=growth_slider.value,
            min_ebitda_margin=ebitda_slider.value if not profitable_checkbox.value else 0,
            industries=list(industry_select.value),
            require_profitability=profitable_checkbox.value,
            exclude_distressed=exclude_distressed.value
        )
        
        current_results = results
        
        # Display results summary
        print(f"\n✓ Found {len(results)} targets")
        
        if len(results) > 0:
            print(f"\nTop 5 Targets:")
            for idx, row in results.head(5).iterrows():
                print(f"{row['rank']}. {row['name']} - Score: {row['total_score']:.1f}")
            
            # Create visualization
            fig = create_target_scatter(
                results.head(20),
                title=f"Top {min(20, len(results))} Targets"
            )
            fig.show()
        else:
            print("No targets found. Try adjusting criteria.")

run_button.on_click(on_run_clicked)

# Display controls
print("SCREENING CONTROLS")
print("="*50)

controls_box = widgets.VBox([
    widgets.HTML("<h3>Financial Criteria</h3>"),
    revenue_slider,
    growth_slider,
    ebitda_slider,
    widgets.HTML("<h3>Industry & Filters</h3>"),
    industry_select,
    widgets.HBox([profitable_checkbox, exclude_distressed]),
    widgets.HTML("<br>"),
    run_button
])

display(controls_box)
display(output)

## 2. Dynamic Filtering & Sorting

Filter and sort results interactively after screening.

In [ ]:
# Create filtering controls
score_filter = widgets.FloatRangeSlider(
    value=[50, 100],
    min=0,
    max=100,
    step=5,
    description='Score Range:',
    style=style,
    layout=widgets.Layout(width='400px')
)

sort_by = widgets.Dropdown(
    options=['total_score', 'strategic_score', 'valuation_score', 'revenue_ttm', 'revenue_growth_yoy'],
    value='total_score',
    description='Sort By:',
    style=style
)

show_top_n = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=5,
    description='Show Top:',
    style=style
)

filter_output = widgets.Output()

def update_filtered_view(change):
    with filter_output:
        clear_output(wait=True)
        
        if current_results is not None and len(current_results) > 0:
            # Apply filters
            min_score, max_score = score_filter.value
            filtered = current_results[
                (current_results['total_score'] >= min_score) &
                (current_results['total_score'] <= max_score)
            ]
            
            # Sort
            filtered = filtered.sort_values(sort_by.value, ascending=False)
            
            # Limit
            filtered = filtered.head(show_top_n.value)
            
            if len(filtered) > 0:
                # Display table
                display_cols = ['rank', 'name', 'industry', 'revenue_ttm', 
                               'revenue_growth_yoy', 'total_score']
                display_df = filtered[display_cols].copy()
                display_df['revenue_ttm'] = display_df['revenue_ttm'].apply(lambda x: f"${x/1e6:.1f}M")
                display_df['revenue_growth_yoy'] = display_df['revenue_growth_yoy'].apply(lambda x: f"{x:.1%}")
                display_df['total_score'] = display_df['total_score'].apply(lambda x: f"{x:.1f}")
                
                display(HTML(display_df.to_html(index=False, classes='table table-striped')))
            else:
                print("No targets match filter criteria.")
        else:
            print("Run screening first to see results.")

# Connect controls
score_filter.observe(update_filtered_view, names='value')
sort_by.observe(update_filtered_view, names='value')
show_top_n.observe(update_filtered_view, names='value')

# Display filter controls
print("\nFILTER & SORT CONTROLS")
print("="*50)

filter_box = widgets.VBox([
    score_filter,
    sort_by,
    show_top_n
])

display(filter_box)
display(filter_output)

## 3. Target Deep Dive Selector

Select any target for detailed analysis.

In [ ]:
# Target selector
target_dropdown = widgets.Dropdown(
    options=['Run screening first'],
    description='Select Target:',
    style=style,
    layout=widgets.Layout(width='400px')
)

analyze_button = widgets.Button(
    description='Analyze Target',
    button_style='success',
    icon='chart-line'
)

analysis_output = widgets.Output()

def update_target_list(b):
    if current_results is not None and len(current_results) > 0:
        target_options = [(f"{row['name']} (Score: {row['total_score']:.1f})", idx) 
                         for idx, row in current_results.head(20).iterrows()]
        target_dropdown.options = target_options

def analyze_target(b):
    with analysis_output:
        clear_output(wait=True)
        
        if current_results is not None and target_dropdown.value != 'Run screening first':
            target_idx = target_dropdown.value
            target = current_results.loc[target_idx]
            
            print(f"ANALYZING: {target['name']}")
            print("="*60)
            
            # Company overview
            print("\nCompany Overview:")
            print(f"  Industry: {target['industry']} - {target['sub_industry']}")
            print(f"  Revenue: ${target['revenue_ttm']/1e6:.1f}M")
            print(f"  Growth: {target['revenue_growth_yoy']:.1%}")
            print(f"  EBITDA Margin: {target.get('ebitda_margin', 0):.1%}")
            print(f"  Employees: {target['employees']:,}")
            
            # Scores breakdown
            scores = {
                'Strategic': target['strategic_score'],
                'Valuation': target['valuation_score'],
                'Financial': target['financial_score'],
                'Risk': target['risk_score']
            }
            
            fig = create_score_breakdown(scores, target['name'])
            fig.show()
            
            # Quick valuation
            print("\nQuick Valuation:")
            dcf = valuation_engine.dcf_valuation(target)
            print(f"  Current EV: ${target['enterprise_value']/1e6:.1f}M")
            print(f"  DCF Value: ${dcf['value']/1e6:.1f}M")
            print(f"  Implied Return: {(dcf['value']/target['enterprise_value'] - 1):.1%}")
            
            # Recommendation
            print(f"\nRecommendation: {target['recommendation']}")

run_button.on_click(update_target_list)
analyze_button.on_click(analyze_target)

print("\nTARGET ANALYSIS")
print("="*50)

analysis_box = widgets.VBox([
    widgets.HTML("<p>Select a target for detailed analysis:</p>"),
    target_dropdown,
    analyze_button
])

display(analysis_box)
display(analysis_output)

## 4. Portfolio Builder

Build and analyze a portfolio of acquisition targets.

In [ ]:
# Portfolio management
portfolio = []
portfolio_output = widgets.Output()

add_to_portfolio = widgets.Button(
    description='Add to Portfolio',
    button_style='info',
    icon='plus'
)

clear_portfolio = widgets.Button(
    description='Clear Portfolio',
    button_style='warning',
    icon='trash'
)

analyze_portfolio = widgets.Button(
    description='Analyze Portfolio',
    button_style='primary',
    icon='chart-bar'
)

def add_target_to_portfolio(b):
    global portfolio
    
    if current_results is not None and target_dropdown.value != 'Run screening first':
        target_idx = target_dropdown.value
        target = current_results.loc[target_idx]
        
        # Check if already in portfolio
        if target_idx not in [t['idx'] for t in portfolio]:
            portfolio.append({
                'idx': target_idx,
                'name': target['name'],
                'industry': target['industry'],
                'revenue': target['revenue_ttm'],
                'score': target['total_score']
            })
            
            with portfolio_output:
                clear_output(wait=True)
                print(f"Added {target['name']} to portfolio")
                print(f"Portfolio size: {len(portfolio)} targets")

def clear_portfolio_func(b):
    global portfolio
    portfolio = []
    with portfolio_output:
        clear_output(wait=True)
        print("Portfolio cleared")

def analyze_portfolio_func(b):
    with portfolio_output:
        clear_output(wait=True)
        
        if len(portfolio) > 0:
            print("PORTFOLIO ANALYSIS")
            print("="*60)
            
            # Create portfolio DataFrame
            portfolio_df = pd.DataFrame(portfolio)
            
            # Summary statistics
            print(f"\nPortfolio Size: {len(portfolio)} targets")
            print(f"Total Value: ${portfolio_df['revenue'].sum()/1e6:.1f}M")
            print(f"Average Score: {portfolio_df['score'].mean():.1f}")
            
            # Industry breakdown
            print("\nIndustry Distribution:")
            for industry, count in portfolio_df['industry'].value_counts().items():
                print(f"  {industry}: {count} targets")
            
            # Visualize portfolio
            fig = px.treemap(
                portfolio_df,
                path=['industry', 'name'],
                values='revenue',
                color='score',
                color_continuous_scale='RdYlGn',
                title='Portfolio Composition'
            )
            fig.show()
            
            # Display portfolio table
            display_df = portfolio_df[['name', 'industry', 'revenue', 'score']].copy()
            display_df['revenue'] = display_df['revenue'].apply(lambda x: f"${x/1e6:.1f}M")
            display_df['score'] = display_df['score'].apply(lambda x: f"{x:.1f}")
            
            print("\nPortfolio Targets:")
            display(HTML(display_df.to_html(index=False, classes='table table-striped')))
        else:
            print("Portfolio is empty. Add targets first.")

add_to_portfolio.on_click(add_target_to_portfolio)
clear_portfolio.on_click(clear_portfolio_func)
analyze_portfolio.on_click(analyze_portfolio_func)

print("\nPORTFOLIO BUILDER")
print("="*50)

portfolio_box = widgets.HBox([
    add_to_portfolio,
    clear_portfolio,
    analyze_portfolio
])

display(portfolio_box)
display(portfolio_output)

## 5. Scenario Comparison

Compare different screening scenarios side by side.

In [ ]:
# Scenario comparison
scenarios = {
    'High Growth': {
        'min_revenue': 10_000_000,
        'max_revenue': 100_000_000,
        'min_growth': 0.50,
        'industries': ['SaaS', 'Fintech']
    },
    'Profitable': {
        'min_revenue': 50_000_000,
        'max_revenue': 500_000_000,
        'min_growth': 0.15,
        'require_profitability': True
    },
    'Undervalued': {
        'min_revenue': 20_000_000,
        'max_revenue': 200_000_000,
        'min_growth': 0.10
    }
}

scenario_output = widgets.Output()

compare_button = widgets.Button(
    description='Compare Scenarios',
    button_style='primary',
    icon='balance-scale'
)

def compare_scenarios(b):
    with scenario_output:
        clear_output(wait=True)
        
        print("SCENARIO COMPARISON")
        print("="*60)
        
        comparison_results = []
        
        for scenario_name, criteria in scenarios.items():
            print(f"\nRunning: {scenario_name}...")
            
            results = screener.screen(**criteria)
            
            # Focus on undervalued in the Undervalued scenario
            if scenario_name == 'Undervalued':
                results = results[results['is_undervalued'] == True]
            
            comparison_results.append({
                'Scenario': scenario_name,
                'Targets Found': len(results),
                'Avg Score': results['total_score'].mean() if len(results) > 0 else 0,
                'Avg Revenue': results['revenue_ttm'].mean() / 1e6 if len(results) > 0 else 0,
                'Avg Growth': results['revenue_growth_yoy'].mean() if len(results) > 0 else 0,
                'Top Target': results.iloc[0]['name'] if len(results) > 0 else 'N/A'
            })
        
        # Display comparison
        comparison_df = pd.DataFrame(comparison_results)
        
        # Format for display
        display_df = comparison_df.copy()
        display_df['Avg Score'] = display_df['Avg Score'].apply(lambda x: f"{x:.1f}")
        display_df['Avg Revenue'] = display_df['Avg Revenue'].apply(lambda x: f"${x:.1f}M")
        display_df['Avg Growth'] = display_df['Avg Growth'].apply(lambda x: f"{x:.1%}")
        
        print("\nScenario Results:")
        display(HTML(display_df.to_html(index=False, classes='table table-striped')))
        
        # Visualize comparison
        fig = px.bar(
            comparison_df,
            x='Scenario',
            y='Targets Found',
            color='Avg Score',
            title='Scenario Comparison - Target Count vs Quality',
            color_continuous_scale='RdYlGn'
        )
        fig.show()

compare_button.on_click(compare_scenarios)

print("\nSCENARIO ANALYSIS")
print("="*50)
print("\nCompare pre-defined screening scenarios:")
for name, criteria in scenarios.items():
    print(f"  • {name}: {criteria.get('min_growth', 0):.0%}+ growth")

display(compare_button)
display(scenario_output)

## 6. Real-Time Market Monitor

Simulate real-time monitoring of target companies.

In [ ]:
import time
import random
from datetime import datetime

# Monitoring controls
monitor_output = widgets.Output()
monitoring = False

start_monitor = widgets.Button(
    description='Start Monitoring',
    button_style='success',
    icon='play'
)

stop_monitor = widgets.Button(
    description='Stop Monitoring',
    button_style='danger',
    icon='stop'
)

def simulate_market_event():
    """Generate a simulated market event"""
    events = [
        {'type': 'Price Movement', 'icon': '📊'},
        {'type': 'News Alert', 'icon': '📰'},
        {'type': 'M&A Rumor', 'icon': '🔔'},
        {'type': 'Earnings Update', 'icon': '💰'},
        {'type': 'Management Change', 'icon': '👔'}
    ]
    
    if current_results is not None and len(current_results) > 0:
        target = current_results.sample(1).iloc[0]
        event = random.choice(events)
        
        # Generate event details
        if event['type'] == 'Price Movement':
            change = random.uniform(-0.15, 0.15)
            detail = f"Stock {'up' if change > 0 else 'down'} {abs(change):.1%}"
        elif event['type'] == 'News Alert':
            detail = random.choice(['New product launch', 'Partnership announced', 'Expansion plans'])
        elif event['type'] == 'M&A Rumor':
            detail = 'Potential acquirer interest reported'
        elif event['type'] == 'Earnings Update':
            detail = f"Q3 earnings {'beat' if random.random() > 0.5 else 'miss'} expectations"
        else:
            detail = random.choice(['New CEO appointed', 'CFO departure', 'Board changes'])
        
        return {
            'time': datetime.now().strftime('%H:%M:%S'),
            'company': target['name'],
            'event': event['type'],
            'icon': event['icon'],
            'detail': detail
        }
    return None

def start_monitoring(b):
    global monitoring
    monitoring = True
    
    with monitor_output:
        clear_output(wait=True)
        print("MARKET MONITOR ACTIVE")
        print("="*60)
        print("Monitoring targets for significant events...\n")
        
        event_count = 0
        while monitoring and event_count < 10:  # Limit to 10 events for demo
            time.sleep(random.uniform(2, 5))  # Random delay
            
            if not monitoring:
                break
            
            event = simulate_market_event()
            if event:
                print(f"{event['icon']} [{event['time']}] {event['company']}")
                print(f"   {event['event']}: {event['detail']}\n")
                event_count += 1
        
        if event_count >= 10:
            print("\n[Monitor paused after 10 events]")
            monitoring = False

def stop_monitoring(b):
    global monitoring
    monitoring = False
    with monitor_output:
        print("\n[Monitoring stopped]")

start_monitor.on_click(start_monitoring)
stop_monitor.on_click(stop_monitoring)

print("\nMARKET MONITOR")
print("="*50)
print("Simulate real-time monitoring of target companies:")

monitor_box = widgets.HBox([start_monitor, stop_monitor])
display(monitor_box)
display(monitor_output)

## 7. Export & Reporting

Export results and generate reports.

In [ ]:
# Export controls
export_output = widgets.Output()

export_excel = widgets.Button(
    description='Export to Excel',
    button_style='success',
    icon='file-excel'
)

generate_report = widgets.Button(
    description='Generate Report',
    button_style='primary',
    icon='file-text'
)

def export_to_excel(b):
    with export_output:
        clear_output(wait=True)
        
        if current_results is not None and len(current_results) > 0:
            filename = f'../data/meridian_results_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'
            
            # Prepare export data
            export_df = current_results.head(50).copy()
            
            # Format columns
            export_df['revenue_ttm'] = export_df['revenue_ttm'].apply(lambda x: x/1e6)
            export_df['enterprise_value'] = export_df['enterprise_value'].apply(lambda x: x/1e6)
            
            # Save to Excel
            export_df.to_excel(filename, index=False)
            
            print(f"✓ Exported {len(export_df)} targets to:")
            print(f"  {filename}")
        else:
            print("No results to export. Run screening first.")

def generate_report_func(b):
    with export_output:
        clear_output(wait=True)
        
        if current_results is not None and len(current_results) > 0:
            print("Generating executive report...")
            
            # Generate report using analyzer
            report = analyzer.generate_executive_summary(
                current_results.head(20),
                include_recommendations=True
            )
            
            # Save report
            filename = f'../data/executive_report_{datetime.now().strftime("%Y%m%d_%H%M")}.txt'
            with open(filename, 'w') as f:
                f.write(report)
            
            print(f"\n✓ Report generated and saved to:")
            print(f"  {filename}")
            
            # Display preview
            print("\nReport Preview:")
            print("="*60)
            print(report[:500] + "...\n[Report continues]")
        else:
            print("No results to report. Run screening first.")

export_excel.on_click(export_to_excel)
generate_report.on_click(generate_report_func)

print("\nEXPORT & REPORTING")
print("="*50)

export_box = widgets.HBox([export_excel, generate_report])
display(export_box)
display(export_output)

## 8. Dashboard Summary

View key metrics and statistics at a glance.

In [ ]:
# Summary dashboard
def create_summary_dashboard():
    if current_results is not None and len(current_results) > 0:
        # Calculate metrics
        total_targets = len(current_results)
        avg_score = current_results['total_score'].mean()
        immediate_action = len(current_results[current_results['category'] == 'Immediate Action'])
        undervalued = current_results['is_undervalued'].sum()
        
        # Create dashboard
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Score Distribution', 'Industry Breakdown', 
                          'Growth vs Profitability', 'Valuation Distribution'),
            specs=[[{'type': 'histogram'}, {'type': 'pie'}],
                  [{'type': 'scatter'}, {'type': 'box'}]]
        )
        
        # Score distribution
        fig.add_trace(
            go.Histogram(x=current_results['total_score'], nbinsx=20, name='Score'),
            row=1, col=1
        )
        
        # Industry pie
        industry_counts = current_results['industry'].value_counts()
        fig.add_trace(
            go.Pie(labels=industry_counts.index, values=industry_counts.values),
            row=1, col=2
        )
        
        # Growth vs profitability
        fig.add_trace(
            go.Scatter(
                x=current_results['revenue_growth_yoy'],
                y=current_results['ebitda_margin'],
                mode='markers',
                marker=dict(color=current_results['total_score'], colorscale='RdYlGn'),
                showlegend=False
            ),
            row=2, col=1
        )
        
        # Valuation box plot
        fig.add_trace(
            go.Box(y=current_results['implied_revenue_multiple'], name='Multiple'),
            row=2, col=2
        )
        
        fig.update_layout(height=700, showlegend=False, title_text="Dashboard Summary")
        fig.show()
        
        # Display metrics
        print("\nKEY METRICS")
        print("="*60)
        print(f"Total Targets: {total_targets}")
        print(f"Average Score: {avg_score:.1f}")
        print(f"Immediate Action: {immediate_action}")
        print(f"Undervalued: {undervalued}")
        print(f"\nTop Industries:")
        for industry, count in industry_counts.head(3).items():
            print(f"  • {industry}: {count} targets")
    else:
        print("No data available. Run screening first.")

summary_button = widgets.Button(
    description='Update Dashboard',
    button_style='primary',
    icon='dashboard'
)

summary_output = widgets.Output()

def update_summary(b):
    with summary_output:
        clear_output(wait=True)
        create_summary_dashboard()

summary_button.on_click(update_summary)

print("\nDASHBOARD SUMMARY")
print("="*50)

display(summary_button)
display(summary_output)

## Summary

This interactive dashboard provides:

1. **Real-time screening** with adjustable parameters
2. **Dynamic filtering and sorting** of results
3. **Target deep-dive analysis** with one click
4. **Portfolio building** capabilities
5. **Scenario comparison** tools
6. **Market monitoring** simulation
7. **Export and reporting** functionality
8. **Executive dashboard** with key metrics

The platform enables rapid, data-driven M&A decision making with full interactivity.

---
*Meridian M&A Intelligence Platform - Interactive Dashboard*